# Reading, describing and preparing a dataset: the Ames houses

This lab puts the chapter "A Tour of pandas" into practice on real data: the 1,460 house sales of Ames (Iowa) from 2006 to 2010, described by 79 variables (`ames/train.csv`, every variable and each of its codes being explained in `ames/data_description.txt`), then, to read an Excel workbook, INSEE's population estimates by French département (`france-population/`).

The exercises follow one another: each one starts from the `houses` DataFrame as the previous ones left it. The Additional hints give the functions, the parameters and a skeleton of the code; the Solution, to run before moving on if you open it, defines everything the next exercises use.

The next cell downloads both datasets into the current directory, the one after imports the libraries.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh
![ -d ames ] || bash /tmp/ghsparse.sh ames
![ -d france-population ] || bash /tmp/ghsparse.sh france-population

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 1. The working environment

Colab runs this notebook on a Linux virtual machine that has its own Python and a few hundred preinstalled packages (pandas, NumPy, pyarrow, matplotlib…), at versions Google picks and updates without notice; whatever you install or write there is gone when you disconnect. The course's Codespace reproduces the same environment. To recreate it on your machine, you note the versions, then pin them in a `uv` project (frame "Virtual Environment with `uv`"):

```console
$ uv init --python 3.13 analysis && cd analysis
$ uv add pandas==2.2.3 numpy==2.1.3 …    # the command you are about to build
$ uv run --with jupyter jupyter lab       # this notebook, in that environment
```

Print the pandas version (`pd.__version__`). Write `pd.show_versions`'s report to the file `versions.json`, read it back with the `json` module, print the Python version it gives, and build the dictionary `versions` of the versions of `pandas`, `numpy`, `pyarrow`, `openpyxl`, `xlrd` and `matplotlib`. A `None` value means a missing package: there must be none, as pyarrow is needed for Parquet (exercise 9) and xlrd for INSEE's `.xls` workbook. Finally, print the `uv add` command that installs exactly these versions.

**Useful documentation**

- [`pandas.show_versions`](https://pandas.pydata.org/docs/reference/api/pandas.show_versions.html)
- [`json.load`](https://docs.python.org/3/library/json.html#json.load)

In [ ]:
# Your code here

### Additional hints

`pd.show_versions(as_json="versions.json")` writes the report to that file instead of printing it. Read back with `json.load`, it is a dictionary with two keys: `"system"` (including `"python"`, the Python version) and `"dependencies"` (package name → version, or `None` when it isn't installed). A dictionary comprehension extracts the six packages, and `" ".join(...)` assembles the command from each package's string `f"{name}=={version}"` (`versions.items()`), of the form `uv add pandas==2.2.3 numpy==2.1.3 pyarrow==…` (Colab's versions in October 2026).

```python
print(pd.__version__)
pd.show_versions(as_json="versions.json")
with open("versions.json") as f:
  report = json.load(f)
print("Python", report["system"]["python"])
packages = ["pandas", "numpy", "pyarrow", "openpyxl", "xlrd", "matplotlib"]
versions = {name: ... for name in packages}
print(versions)
print("uv add " + " ".join(...))
```

### Solution

In [ ]:
print(pd.__version__)
pd.show_versions(as_json="versions.json")
with open("versions.json") as f:
  report = json.load(f)
print("Python", report["system"]["python"])
packages = ["pandas", "numpy", "pyarrow", "openpyxl", "xlrd", "matplotlib"]
versions = {name: report["dependencies"][name] for name in packages}
print(versions)
print("uv add " + " ".join(f"{name}=={version}" for name, version in versions.items()))

## 2. Reading the CSV without losing information

In `ames/train.csv`, an empty field is an unknown value, but `NA` often means "none": `Alley` is `NA` for "No alley access", `PoolQC` for "No Pool" (see `ames/data_description.txt`). Yet by default `read_csv` reads `NA` (like `None`, `n/a`, `NaN`…) as missing: these absences then get mixed up with true unknowns.

- Read the file into `houses`, indexed by `Id`, so that only the empty fields are missing, and that `MSSubClass` (a dwelling type code, not a quantity, despite its values 20, 60…) has the `category` type.
- Check with `value_counts(dropna=False)` that `Alley` has an `NA` category, and compare the total number of missing values of `houses` with that of a default read.
- Finally, read into `preview` only the columns `Id`, `Neighborhood` and `SalePrice`, and compare the memory both DataFrames use (`memory_usage(deep=True)`).

**Useful documentation**

- [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

In [ ]:
# Your code here

### Additional hints

`keep_default_na=False` empties the list of strings read as missing, and `na_values=[""]` puts only the empty field back in it. `dtype={"MSSubClass": "category"}` sets a column's type as it is read. Expected: 357 missing values, against 7,829 by default, and for `Alley` 1,369 `NA`, 50 `Grvl` and 41 `Pave`. With `usecols=`, the `index_col` column must be in the list. `memory_usage(deep=True).sum()`: 3.7 MB for `houses`, 0.1 MB for `preview`.

```python
houses = pd.read_csv(
  "ames/train.csv",
  index_col="Id",
  keep_default_na=...,
  na_values=...,
  dtype=...,
)
print(houses["Alley"].value_counts(dropna=False))
default = pd.read_csv("ames/train.csv", index_col="Id")
print(houses.isna().sum().sum(), ...)
preview = pd.read_csv("ames/train.csv", usecols=..., index_col="Id")
print(houses.memory_usage(deep=True).sum(), ...)
```

### Solution

In [ ]:
houses = pd.read_csv(
  "ames/train.csv",
  index_col="Id",
  keep_default_na=False,
  na_values=[""],
  dtype={"MSSubClass": "category"},
)
print(houses["Alley"].value_counts(dropna=False))
default = pd.read_csv("ames/train.csv", index_col="Id")
print(houses.isna().sum().sum(), default.isna().sum().sum())
preview = pd.read_csv(
  "ames/train.csv", usecols=["Id", "Neighborhood", "SalePrice"], index_col="Id"
)
print(houses.memory_usage(deep=True).sum(), preview.memory_usage(deep=True).sum())

## 3. Describing the dataset

Draw a portrait of `houses`:

- types and real memory with `info`, then the number of columns of each type (`dtypes.value_counts()`);
- `describe` of the text columns: which neighborhood (`Neighborhood`) has the most sales?
- `describe` of the columns `LotArea`, `GrLivArea`, `YearBuilt`, `OverallQual` and `SalePrice`: their orders of magnitude, and the highest price;
- the number of distinct values of each column (`nunique`), sorted: which columns are nearly constant (look at the `value_counts` of `Utilities`)? Among the numeric columns (`select_dtypes`), which ones have a dozen values at most, grades or months rather than quantities?

**Useful documentation**

- [`DataFrame.nunique`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html)

In [ ]:
# Your code here

### Additional hints

`info(memory_usage="deep")` counts the strings' real memory (3.6 MB): 43 `object` columns, 33 `int64`, 3 `float64` and 1 `category`. `describe(include="object")` gives each text column's `count`, `unique`, `top` (the most frequent value) and `freq` (its count): `NAmes`, 225 sales; `.T` puts one row per column, easier to read. The highest price is 755,000 dollars. Sorted, `nunique()` starts with `Street`, `Utilities` and `CentralAir` (two values): `Utilities` is `AllPub` for 1,459 houses out of 1,460, it tells nothing. Among the numbers, `OverallQual` (a grade from 1 to 10), `YrSold` or `GarageCars` have 10 values at most.

```python
houses.info(memory_usage=...)
print(houses.dtypes.value_counts())
print(houses.describe(include=...).T)
print(
  houses[["LotArea", "GrLivArea", "YearBuilt", "OverallQual", "SalePrice"]].describe().T
)
print(houses.nunique().sort_values().head())
print(houses["Utilities"].value_counts())
print(houses.select_dtypes(...).nunique().sort_values().head(12))
```

### Solution

In [ ]:
houses.info(memory_usage="deep")
print(houses.dtypes.value_counts())
print(houses.describe(include="object").T)
print(
  houses[["LotArea", "GrLivArea", "YearBuilt", "OverallQual", "SalePrice"]].describe().T
)
print(houses.nunique().sort_values().head())
print(houses["Utilities"].value_counts())
print(houses.select_dtypes("number").nunique().sort_values().head(12))

## 4. Missing values

The only missing values left in `houses` are the file's empty fields.

- **Profile**: the share of missing values of each column (`isna().mean()`), sorted, for the columns that have some only.
- **Structural absence**: is the garage's year `GarageYrBlt` missing exactly for the houses without a garage (`GarageType` is `"NA"`)? Check it with `pd.crosstab`. Do the same for the basement, between `BsmtQual` and `BsmtExposure` being `"NA"`: one house lies off the diagonal. Which one (its `Id`), and what does its `NA` mean then? Replace it with a true missing value, `np.nan`, in a single assignment with `.loc`.
- **Imputation**: the street frontage `LotFrontage` depends on the neighborhood. Replace its missing values with the median of their neighborhood (`groupby` then `transform`), and check that none is left.

**Useful documentation**

- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)
- [`SeriesGroupBy.transform`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.transform.html)

In [ ]:
# Your code here

### Additional hints

`missing[missing > 0]` keeps the columns concerned only: `LotFrontage` (17.7%), `GarageYrBlt` (5.5%), `MasVnrType` and `MasVnrArea` (0.5%), `Electrical` (one house). `pd.crosstab(a, b)` crosses two boolean `Series`: a diagonal table (1,379 and 81) shows that both are missing together, and that the 81 missing years have nothing to impute. For the basement, the house off the diagonal has a basement but an `NA` exposure: an unknown, not an absence. Combine both masks with `~` and `&`, and read the index: it is house 949. `groupby("Neighborhood")["LotFrontage"].transform("median")` returns a `Series` aligned on `houses` (the median of each house's neighborhood), which `fillna` accepts: house 8, in the `NWAmes` neighborhood, gets 80. To train a model, these medians would be computed on the training set only (frame "Imputing Without Leakage, Filling a Series").

```python
missing = houses.isna().mean().sort_values(ascending=False)
print(missing[...])
print(pd.crosstab(houses["GarageYrBlt"].isna(), houses["GarageType"] == "NA"))
no_basement = houses["BsmtQual"] == "NA"
exposure_na = ...
print(pd.crosstab(no_basement, exposure_na))
print(houses.index[...])
houses.loc[..., "BsmtExposure"] = np.nan
neighborhood_median = houses.groupby("Neighborhood")["LotFrontage"].transform(...)
houses["LotFrontage"] = houses["LotFrontage"].fillna(...)
print(houses["LotFrontage"].isna().sum(), houses.loc[8, "LotFrontage"])
```

### Solution

In [ ]:
missing = houses.isna().mean().sort_values(ascending=False)
print(missing[missing > 0])
print(pd.crosstab(houses["GarageYrBlt"].isna(), houses["GarageType"] == "NA"))
no_basement = houses["BsmtQual"] == "NA"
exposure_na = houses["BsmtExposure"] == "NA"
print(pd.crosstab(no_basement, exposure_na))
print(houses.index[~no_basement & exposure_na])
houses.loc[~no_basement & exposure_na, "BsmtExposure"] = np.nan
neighborhood_median = houses.groupby("Neighborhood")["LotFrontage"].transform("median")
houses["LotFrontage"] = houses["LotFrontage"].fillna(neighborhood_median)
print(houses["LotFrontage"].isna().sum(), houses.loc[8, "LotFrontage"])

## 5. Types: ordered quality scales

`ExterQual`, `ExterCond`, `HeatingQC` and `KitchenQual` grade a quality on the scale `Po` < `Fa` < `TA` < `Gd` < `Ex` (from "Poor" to "Excellent"). As text, `"Ex" < "Gd"` is true (alphabetical order) and sorting mixes the levels up.

- Create the type `quality`, an ordered categorical on this scale (`pd.CategoricalDtype`), and convert the four columns to it.
- Count the houses whose exterior quality is at least good (`ExterQual` at least `"Gd"`), and print the median price for each level of `ExterQual`: it must increase with the quality.
- Measure, without changing `houses`, the memory its `object` columns use before and after `astype("category")`.

**Useful documentation**

- [`pandas.CategoricalDtype`](https://pandas.pydata.org/docs/reference/api/pandas.CategoricalDtype.html)

In [ ]:
# Your code here

### Additional hints

`pd.CategoricalDtype([...], ordered=True)` defines the type, from the lowest level to the highest; `houses[columns].astype(quality)` converts several columns at once. A comparison with a string then follows the categories' order: 540 houses. A `groupby` on a categorical column takes `observed=True` (the levels present only: no house has `Po`) and orders the groups along the scale, from 82,250 dollars (`Fa`) to 364,606.5 dollars (`Ex`). `select_dtypes("object")` gives the text columns, and `memory_usage(deep=True).sum()` their memory: 3 MB as `object`, 0.1 MB as `category`.

```python
quality = pd.CategoricalDtype(..., ordered=True)
columns = ["ExterQual", "ExterCond", "HeatingQC", "KitchenQual"]
houses[columns] = ...
print((houses["ExterQual"] >= ...).sum())
print(houses.groupby("ExterQual", observed=True)["SalePrice"].median())
texts = houses.select_dtypes("object")
print(texts.memory_usage(deep=True).sum(), ...)
```

### Solution

In [ ]:
quality = pd.CategoricalDtype(["Po", "Fa", "TA", "Gd", "Ex"], ordered=True)
columns = ["ExterQual", "ExterCond", "HeatingQC", "KitchenQual"]
houses[columns] = houses[columns].astype(quality)
print((houses["ExterQual"] >= "Gd").sum())
print(houses.groupby("ExterQual", observed=True)["SalePrice"].median())
texts = houses.select_dtypes("object")
print(
  texts.memory_usage(deep=True).sum(),
  texts.astype("category").memory_usage(deep=True).sum(),
)

## 6. Strings: reconciling two columns

`Exterior1st` and `Exterior2nd` give the house's main exterior covering and the second one (the same when there is only one). The documentation gives them the same codes, but the file spells some of them differently in `Exterior2nd`.

- Print the codes of `Exterior2nd` that `Exterior1st` doesn't have, with their counts (`isin`), and find in `ames/data_description.txt` the code they should have.
- Fix `Exterior2nd` in `houses` with a dictionary of replacements (`replace`), and count the houses with a single covering (both columns equal) before and after the fix.
- With the `.str` accessor, count the houses with wood on at least one of their two coverings (the codes starting with `Wd`).

**Useful documentation**

- [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)
- [`Series.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)
- [`Series.str.startswith`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.startswith.html)

In [ ]:
# Your code here

### Additional hints

`~houses["Exterior2nd"].isin(houses["Exterior1st"])` is true for the codes the other column doesn't have: `CmentBd` (60 houses), `Wd Shng` (38), `Brk Cmn` (7), and `Other` (1, a code the documentation gives to `Exterior2nd` only). They are spelled `CemntBd`, `WdShing` and `BrkComm` in `Exterior1st`. `Series.replace` with a dictionary `{old: new}` replaces whole values (`.str.replace` replaces pieces of strings). Expected: 1,245 single coverings before, 1,323 after. `.str.startswith("Wd")` gives a mask per column, and `|` combines them: 263 houses.

```python
absent = ~houses["Exterior2nd"].isin(...)
print(houses.loc[absent, "Exterior2nd"].value_counts())
single_before = (houses["Exterior1st"] == houses["Exterior2nd"]).sum()
fixes = {"Wd Shng": "WdShing", "CmentBd": ..., "Brk Cmn": ...}
houses["Exterior2nd"] = ...
single_after = ...
print(single_before, single_after)
wood = houses["Exterior1st"].str.startswith("Wd") | ...
print(wood.sum())
```

### Solution

In [ ]:
absent = ~houses["Exterior2nd"].isin(houses["Exterior1st"])
print(houses.loc[absent, "Exterior2nd"].value_counts())
single_before = (houses["Exterior1st"] == houses["Exterior2nd"]).sum()
fixes = {"Wd Shng": "WdShing", "CmentBd": "CemntBd", "Brk Cmn": "BrkComm"}
houses["Exterior2nd"] = houses["Exterior2nd"].replace(fixes)
single_after = (houses["Exterior1st"] == houses["Exterior2nd"]).sum()
print(single_before, single_after)
wood = houses["Exterior1st"].str.startswith("Wd") | houses[
  "Exterior2nd"
].str.startswith("Wd")
print(wood.sum())

## 7. Dates: the sale date

The file gives the month (`MoSold`) and the year (`YrSold`) of each sale.

- Build the column `sale_date`, the 1st of the sale's month, with `pd.to_datetime`.
- Compute in `monthly` the number of sales and the median price of each month (`resample("ME")`): how many months does the dataset span, and which one has the most sales?
- Plot the number of sales per month and its 12-month rolling mean (`rolling`): what does each of the two curves show?
- Add the house's age at the sale, `age_at_sale` (the year of `sale_date` minus `YearBuilt`), and cross the houses aged 0 or 1 with the sales of new houses (`SaleType` is `"New"`) with `pd.crosstab`.

**Useful documentation**

- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`DataFrame.resample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)
- [`Resampler.aggregate`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.aggregate.html)

In [ ]:
# Your code here

### Additional hints

`pd.to_datetime` accepts a dictionary (or a DataFrame) of `year`, `month` and `day` columns, a single value standing for every row: `dict(year=..., month=..., day=1)`. `houses.resample("ME", on="sale_date")` groups by month on a column, leaving the index alone; `.agg(sales=("SalePrice", "size"), median_price=("SalePrice", "median"))` names the result's columns. Expected: 55 months, from January 2006 to July 2010, with July 2006 first (67 sales). `monthly["sales"].plot()` plots the series on its date index, and `ax=` draws the second curve on top; `rolling(12).mean()` removes the seasonality (a peak every spring and summer): about 26 sales per month, with no trend; it starts with 11 `NaN`, and July 2010, the dataset's last month, is incomplete (6 sales). `.dt.year` gives a date's year: 117 of the 164 houses aged 0 or 1 are sold new.

```python
houses["sale_date"] = pd.to_datetime(dict(year=..., month=..., day=1))
monthly = houses.resample("ME", on=...).agg(
  sales=("SalePrice", "size"), median_price=...
)
print(len(monthly), monthly["sales"].idxmax(), monthly["sales"].max())
ax = monthly["sales"].plot(label="sales per month")
monthly["sales"].rolling(...).mean().plot(ax=ax, label="12-month mean")
ax.legend()
plt.show()
houses["age_at_sale"] = ... - houses["YearBuilt"]
print(houses["age_at_sale"].describe())
print(pd.crosstab(houses["age_at_sale"] <= 1, ...))
```

### Solution

In [ ]:
houses["sale_date"] = pd.to_datetime(
  dict(year=houses["YrSold"], month=houses["MoSold"], day=1)
)
monthly = houses.resample("ME", on="sale_date").agg(
  sales=("SalePrice", "size"), median_price=("SalePrice", "median")
)
print(len(monthly), monthly["sales"].idxmax(), monthly["sales"].max())
ax = monthly["sales"].plot(label="sales per month")
monthly["sales"].rolling(12).mean().plot(ax=ax, label="12-month mean")
ax.legend()
plt.show()
houses["age_at_sale"] = houses["sale_date"].dt.year - houses["YearBuilt"]
print(houses["age_at_sale"].describe())
print(pd.crosstab(houses["age_at_sale"] <= 1, houses["SaleType"] == "New"))

## 8. Analyzing by variable type

- **Numeric**: the skewness (`skew`) of the price `SalePrice`, then that of its logarithm (`np.log1p`); plot their two histograms side by side (`plot.hist` on a `plt.subplots` grid). Which numeric columns are the most skewed?
- **Numeric by category**: the box plot of the price for each level of `ExterQual` (`sns.boxplot`).
- **Categorical**: per neighborhood, the number of sales, the median price and the median living area (`GrLivArea`), in a single named aggregation (`groupby` then `agg`), sorted by median price; then the mix of dwelling types `BldgType` in each neighborhood, as shares (`pd.crosstab`): which neighborhoods have no single-family house (`1Fam`)?

**Useful documentation**

- [`Series.skew`](https://pandas.pydata.org/docs/reference/api/pandas.Series.skew.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)
- [`DataFrameGroupBy.agg`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.agg.html)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# Your code here

### Additional hints

The price's skewness is 1.88 (a long tail of expensive houses), its logarithm's 0.12: nearly symmetric. `select_dtypes("number").skew()` starts with `MiscVal`, `PoolArea` and `LotArea`, nearly always zero or small, with a few huge values. `fig, axes = plt.subplots(1, 2, figsize=(10, 4))` creates two plots, which `ax=axes[0]` and `ax=axes[1]` designate. `sns.boxplot(data=houses, x="ExterQual", y="SalePrice")` follows the order of exercise 5's categories, `Po` included (empty). A named aggregation reads `agg(name=("column", "function"), ...)`: `NridgHt` is the most expensive neighborhood (315,000 dollars), `MeadowV` the cheapest (88,000 dollars). `normalize="index"` divides each row by its total: `Blueste`, `BrDale`, `MeadowV` and `NPkVill` only have townhouses (`Twnhs`, `TwnhsE`).

```python
price = houses["SalePrice"]
print(price.skew(), np.log1p(price).skew())
print(houses.select_dtypes("number").skew().sort_values(ascending=False).head())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
price.plot.hist(bins=40, ax=axes[0], title="SalePrice")
np.log1p(price).plot.hist(bins=40, ax=..., title="log1p(SalePrice)")
plt.show()
sns.boxplot(data=houses, x=..., y=...)
plt.show()
neighborhoods = (
  houses.groupby("Neighborhood")
  .agg(
    sales=("SalePrice", "size"),
    median_price=...,
    median_area=...,
  )
  .sort_values(...)
)
print(neighborhoods)
shares = pd.crosstab(houses["Neighborhood"], houses["BldgType"], normalize=...)
print(shares[shares["1Fam"] == 0])
```

### Solution

In [ ]:
price = houses["SalePrice"]
print(price.skew(), np.log1p(price).skew())
print(houses.select_dtypes("number").skew().sort_values(ascending=False).head())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
price.plot.hist(bins=40, ax=axes[0], title="SalePrice")
np.log1p(price).plot.hist(bins=40, ax=axes[1], title="log1p(SalePrice)")
plt.show()
sns.boxplot(data=houses, x="ExterQual", y="SalePrice")
plt.show()
neighborhoods = (
  houses.groupby("Neighborhood")
  .agg(
    sales=("SalePrice", "size"),
    median_price=("SalePrice", "median"),
    median_area=("GrLivArea", "median"),
  )
  .sort_values("median_price", ascending=False)
)
print(neighborhoods)
shares = pd.crosstab(houses["Neighborhood"], houses["BldgType"], normalize="index")
print(shares[shares["1Fam"] == 0])

## 9. Other formats: Excel and Parquet

Two independent exercises: an Excel workbook as INSEE publishes it, then a round trip of `houses` through a file.

### 9.1. INSEE's workbook

`france-population/estim-pop-dep-sexe-gca-1975-2023.xls` gives the population of each French département on January 1st, from 1975 to 2023: a sheet of notes, `À savoir`, then one sheet per year, with the two-row header of the frame "Excel: Sheets and Multi-Row Headers" (`header=[3, 4]`, `index_col=[0, 1]`). Build `population`, in long format: one row per département and per year, with the columns `department_code` (text), `year` and `population` (integers), the population being the sheet's total (column `Ensemble`, `Total`).

- Read all the year sheets in a single `read_excel` call, from the list of the workbook's sheets (`pd.ExcelFile`). Why not `sheet_name=None`?
- The codes are text (`01`, `2A`), except those of the overseas départements, which the workbook stores as numbers (`971`): convert them all to `str`.
- Keep the départements only: the bottom of each sheet has total rows (`France métropolitaine`, `DOM`…), an empty row and the source. A département code is `01` to `95`, `2A`, `2B` or `971` to `976` (`.str.fullmatch`).
- Put the years together with `pd.concat`, the year as the key (`keys=`).

Check: 96 départements per year until 1989, 100 from 1990 to 2013, 101 since 2014 (Mayotte), and 68,042,591 inhabitants in 2023, the total of the sheet's "France métropolitaine et DOM" row. What was the population of Paris (`75`) in 1975, and in 2023?

**Useful documentation**

- [`pandas.ExcelFile`](https://pandas.pydata.org/docs/reference/api/pandas.ExcelFile.html)
- [`Series.droplevel`](https://pandas.pydata.org/docs/reference/api/pandas.Series.droplevel.html)
- [`Series.str.fullmatch`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# Your code here

#### Additional hints

`sheet_name=None` would also read `À savoir`, which doesn't have this header: `read_excel` fails (`IndexError`). `with pd.ExcelFile(path) as workbook:` opens the workbook once: `workbook.sheet_names` gives its sheets (keep those whose name is made of digits: `name.isdigit()`), and `pd.read_excel(workbook, sheet_name=years, ...)` returns a `dict` sheet name → DataFrame. In each sheet, `sheet[("Ensemble", "Total")]` is a `Series` indexed by (code, name): `.droplevel(1)` keeps the code only. `dtype=` doesn't apply to index columns: convert the index after reading, `total.index = total.index.astype(str)`. The mask `total.index.str.fullmatch(r"\d{2,3}|2[AB]")` drops the totals, the empty row (`"nan"`) and the source. `pd.concat(totals, keys=[...], names=["year", "department_code"])` stacks the `Series` under a two-level index; there is no `NaN` left, hence `.astype("int64")`, then `.rename("population")` and `.reset_index()` give the long format: 4,850 rows. The codes being text, Paris is found with `department_code == '75'`, not `== 75`: 2,309,074 inhabitants in 1975, 2,102,650 in 2023.

```python
path = "france-population/estim-pop-dep-sexe-gca-1975-2023.xls"
with pd.ExcelFile(path) as workbook:
  years = [name for name in workbook.sheet_names if ...]
  sheets = pd.read_excel(workbook, sheet_name=years, header=[3, 4], index_col=[0, 1])
totals = []
for sheet in sheets.values():
  total = sheet[("Ensemble", "Total")].droplevel(1)
  total.index = ...
  totals.append(total[total.index.str.fullmatch(...)])
population = (
  pd.concat(totals, keys=..., names=["year", "department_code"])
  .astype("int64")
  .rename("population")
  .reset_index()
)
print(population.dtypes)
print(population.groupby("year").size().loc[[1975, 1989, 1990, 2013, 2014, 2023]])
print(population.query("year == 2023")["population"].sum())
print(population.query(...))
```

#### Solution

In [ ]:
path = "france-population/estim-pop-dep-sexe-gca-1975-2023.xls"
with pd.ExcelFile(path) as workbook:
  years = [name for name in workbook.sheet_names if name.isdigit()]
  sheets = pd.read_excel(workbook, sheet_name=years, header=[3, 4], index_col=[0, 1])
totals = []
for sheet in sheets.values():
  total = sheet[("Ensemble", "Total")].droplevel(1)
  total.index = total.index.astype(str)
  totals.append(total[total.index.str.fullmatch(r"\d{2,3}|2[AB]")])
population = (
  pd.concat(
    totals,
    keys=[int(year) for year in years],
    names=["year", "department_code"],
  )
  .astype("int64")
  .rename("population")
  .reset_index()
)
print(population.dtypes)
print(population.groupby("year").size().loc[[1975, 1989, 1990, 2013, 2014, 2023]])
print(population.query("year == 2023")["population"].sum())
print(population.query("department_code == '75' and year in [1975, 2023]"))

### 9.2. The round trip through a file

`houses` now has categorical columns (ordered scales among them), a date and `NA` categories. Write it as CSV (`houses.csv`) and as Parquet (`houses.parquet`), and compare the two files' sizes. Then read each of them back, and compare with `houses`:

- the columns' types, in a DataFrame with three columns (`houses`, `csv`, `parquet`) cut down to the rows where a type differs;
- the number of missing values;
- the whole content (`equals`).

What did the CSV lose on the way?

**Useful documentation**

- [`os.path.getsize`](https://docs.python.org/3/library/os.path.html#os.path.getsize)
- [`DataFrame.equals`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.equals.html)

In [ ]:
# Your code here

#### Additional hints

`to_csv` writes the index (`Id`) with the columns: read it back with `index_col="Id"`. `to_parquet` keeps the index, and `read_parquet` restores it. `os.path.getsize`: 490 kB as CSV, 147 kB as Parquet. A DataFrame built from a dictionary of `Series` (the three `dtypes`) aligns them on their index, the column names. The CSV read back lost six types (`MSSubClass` is an integer again, the quality scales text, `sale_date` text) and reads the `NA` categories as missing again: 7,570 instead of 99. The Parquet read back is identical to `houses` (`equals` is `True`).

```python
houses.to_csv("houses.csv")
houses.to_parquet(...)
print(os.path.getsize("houses.csv"), ...)
csv_back = pd.read_csv("houses.csv", index_col=...)
parquet_back = pd.read_parquet(...)
types = pd.DataFrame({"houses": houses.dtypes, "csv": csv_back.dtypes, "parquet": ...})
print(types[(types["csv"] != types["houses"]) | ...])
print(houses.isna().sum().sum(), ...)
print(csv_back.equals(houses), ...)
```

#### Solution

In [ ]:
houses.to_csv("houses.csv")
houses.to_parquet("houses.parquet")
print(os.path.getsize("houses.csv"), os.path.getsize("houses.parquet"))
csv_back = pd.read_csv("houses.csv", index_col="Id")
parquet_back = pd.read_parquet("houses.parquet")
types = pd.DataFrame(
  {"houses": houses.dtypes, "csv": csv_back.dtypes, "parquet": parquet_back.dtypes}
)
print(types[(types["csv"] != types["houses"]) | (types["parquet"] != types["houses"])])
print(
  houses.isna().sum().sum(),
  csv_back.isna().sum().sum(),
  parquet_back.isna().sum().sum(),
)
print(csv_back.equals(houses), parquet_back.equals(houses))